# Exercise 2 — Review Classifier

## Objective
Build a prompt that classifies a customer review into one category.

## Input
- review_text

## Output
A single category label.

In [31]:
%pip install groq python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [32]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found. Check your .env file.")

client = Groq(api_key=api_key)

MODEL = "openai/gpt-oss-120b"

print("Groq client initialized successfully.")

Groq client initialized successfully.


In [33]:
categories = {
    "Positive": "The review expresses satisfaction or praise.",
    "Negative": "The review expresses dissatisfaction or criticism.",
    "Neutral": "The review is factual or has no clear overall sentiment."
}

print("Review categories:")
for label, definition in categories.items():
    print(f"{label}: {definition}")

Review categories:
Positive: The review expresses satisfaction or praise.
Negative: The review expresses dissatisfaction or criticism.
Neutral: The review is factual or has no clear overall sentiment.


In [34]:
prompt_template = """
You are a review classification assistant.

Classify the review into exactly one of these categories:

Positive: The review expresses satisfaction or praise.
Negative: The review expresses dissatisfaction or criticism.
Neutral: The review is factual, mixed without a clear overall sentiment,
or does not express a clear opinion.

Return only one label:
Positive
Negative
Neutral

Do not explain your answer.
Do not include punctuation or any additional text.

Review:
{review_text}
"""

print("Prompt template created successfully.")

Prompt template created successfully.


In [35]:
def classify_review(review_text):
    if not review_text.strip():
        raise ValueError("Review text cannot be empty.")

    prompt = prompt_template.format(review_text=review_text)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Classify reviews using only the three permitted labels. "
                    "Return exactly one label."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_tokens=20
    )

    result = response.choices[0].message.content.strip()
    result = result.strip("*` \n\t")

    if result not in categories:
        raise ValueError(
            f"Unexpected model output: {result!r}. "
            "Expected Positive, Negative, or Neutral."
        )

    return result

In [38]:
def classify_review(review_text):
    if not review_text.strip():
        raise ValueError("Review text cannot be empty.")

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a sentiment classifier. "
                    "Reply with exactly one word: Positive, Negative, or Neutral."
                )
            },
            {
                "role": "user",
                "content": (
                    "Classify this review.\n"
                    "Positive means favorable sentiment.\n"
                    "Negative means unfavorable sentiment.\n"
                    "Neutral means factual or no clear sentiment.\n\n"
                    f"Review: {review_text}\n\n"
                    "Return exactly one label."
                )
            }
        ],
        temperature=0,
        max_completion_tokens=100
    )

    result = response.choices[0].message.content

    if not result or not result.strip():
        raise ValueError(
            "The model returned an empty response. "
            "Try again or inspect the response's finish_reason."
        )

    result = result.strip().strip("*` .!").capitalize()

    if result not in {"Positive", "Negative", "Neutral"}:
        raise ValueError(f"Unexpected model output: {result!r}")

    return result

In [39]:
review_text = "The product is excellent. It works perfectly and arrived on time."

prediction = classify_review(review_text)

print("Review:", review_text)
print("Predicted category:", prediction)

Review: The product is excellent. It works perfectly and arrived on time.
Predicted category: Positive


In [40]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly one word: Positive"
        }
    ],
    max_completion_tokens=100
)

print("Finish reason:", response.choices[0].finish_reason)
print("Response content:", repr(response.choices[0].message.content))
print("Reasoning:", repr(response.choices[0].message.reasoning))

Finish reason: stop
Response content: 'Positive'
Reasoning: 'User wants reply with exactly one word: "Positive". So we output just "Positive". Ensure no extra characters.'


In [41]:
review_text = (
    "The product stopped working after one day. "
    "The quality is poor and I am disappointed."
)

prediction = classify_review(review_text)

print("Review:", review_text)
print("Predicted category:", prediction)

Review: The product stopped working after one day. The quality is poor and I am disappointed.
Predicted category: Negative


In [42]:
review_text = (
    "The package arrived on Tuesday. "
    "The box contained the product and an instruction manual."
)

prediction = classify_review(review_text)

print("Review:", review_text)
print("Predicted category:", prediction)

Review: The package arrived on Tuesday. The box contained the product and an instruction manual.
Predicted category: Neutral
